In [ ]:
import geopandas as gpd
import pandas as pd
import polars as pl

In [5]:
demanda_scan = pl.scan_parquet(r"outputs/Dados_Tarifa_Zero_2023_2025.parquet")
demanda = (
    demanda_scan
    .select(["data", "linha_blt","zona_emb"]) # Pega só o que precisa
    #.filter(pl.col("coluna_importante_1") > 100)            # Filtra os dados
    #.group_by("coluna_importante_2").agg(pl.len())          # Faz agregações
).collect()
demanda

data,linha_blt,zona_emb
str,str,f64
"""20241225""","""3739-10""",231.0
"""20241225""","""3739-10""",231.0
"""20241225""","""5145-10""",211.0
"""20241225""","""5145-10""",211.0
"""20241225""","""5145-10""",211.0
…,…,…
"""20230226""","""8009-31""",109.0
"""20230226""","""8009-31""",109.0
"""20230226""","""172K-10""",152.0


In [6]:
demanda = demanda.with_columns(
    pl.col("data").str.slice(0, 4).alias("Ano"),
    pl.col("data").str.slice(4, 2).alias("Mês")
)
demanda

data,linha_blt,zona_emb,Ano,Mês
str,str,f64,str,str
"""20241225""","""3739-10""",231.0,"""2024""","""12"""
"""20241225""","""3739-10""",231.0,"""2024""","""12"""
"""20241225""","""5145-10""",211.0,"""2024""","""12"""
"""20241225""","""5145-10""",211.0,"""2024""","""12"""
"""20241225""","""5145-10""",211.0,"""2024""","""12"""
…,…,…,…,…
"""20230226""","""8009-31""",109.0,"""2023""","""02"""
"""20230226""","""8009-31""",109.0,"""2023""","""02"""
"""20230226""","""172K-10""",152.0,"""2023""","""02"""


In [8]:
demanda_mes = (
    demanda.lazy()  # 1. Convert the DataFrame to a LazyFrame
    .group_by(["linha_blt", "zona_emb", "Ano", "Mês"])
    .agg(
        pl.len().alias("N_embarques")
    )
    .collect(engine="streaming")  # 2. Execute the query using the streaming engine
)

C:\Users\9837292\AppData\Local\Temp\ipykernel_60536\1929032341.py:7: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)  # 2. Execute the query using the streaming engine


In [9]:
demanda_mes

linha_blt,zona_emb,Ano,Mês,N_embarques
str,f64,str,str,u32
"""5752-10""",279.0,"""2024""","""12""",32315
"""N501-11""",256.0,"""2024""","""12""",314
"""9784-10""",92.0,"""2024""","""12""",29644
"""N141-11""",122.0,"""2024""","""12""",206
"""351F-10""",166.0,"""2024""","""12""",3098
…,…,…,…,…
"""9012-10""",139.0,"""2023""","""03""",1
"""N841-11""",57.0,"""2023""","""03""",1
"""N504-11""",41.0,"""2023""","""02""",2


In [10]:
demanda_mes.write_parquet("outputs/Dados_Tarifa_Zero_2023_2025_mes.parquet")